# Step 2 — Membuat Database SQLite

Notebook ini akan:

1. Membuat folder `data/`
2. Membuat database `linux_docs.db`
3. Membuat tabel:
   - `pages`
   - `chunks`
   - `crawl_history`
4. Menampilkan daftar tabel untuk memastikan database berhasil dibuat.

> Notebook ini menggunakan modul `sqlite3` bawaan Python, sehingga tidak memerlukan instalasi tambahan.


In [12]:
from pathlib import Path
import sqlite3

# Relative path
DB_PATH = Path("./data/linux_docs.db")

# Pastikan folder data ada
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Database akan dibuat di: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

Database akan dibuat di: data/linux_docs.db


## Membuat tabel

In [13]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS pages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT UNIQUE NOT NULL,
    title TEXT,
    html TEXT,
    markdown TEXT,
    scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    status TEXT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    page_id INTEGER NOT NULL,
    section TEXT,
    chunk_index INTEGER,
    content TEXT NOT NULL,
    token_count INTEGER,
    FOREIGN KEY(page_id) REFERENCES pages(id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS crawl_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT UNIQUE NOT NULL,
    last_crawled TIMESTAMP,
    status_code INTEGER,
    etag TEXT,
    last_modified TEXT
);
""")

conn.commit()

print("Semua tabel berhasil dibuat.")

Semua tabel berhasil dibuat.


## Verifikasi

In [14]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""")

tables = cursor.fetchall()

print("Daftar tabel:")
for (name,) in tables:
    print("-", name)


Daftar tabel:
- chunks
- crawl_history
- pages
- sqlite_sequence


## Penjelasan Singkat

### `pages`
Menyimpan hasil scraping mentah dan hasil parsing.

### `chunks`
Menyimpan hasil pemotongan dokumen menjadi bagian-bagian kecil untuk embedding.

### `crawl_history`
Menyimpan informasi crawl sehingga di masa depan kita dapat melakukan *incremental crawl* tanpa mengunduh ulang halaman yang tidak berubah.


In [15]:
conn.close()
print("Selesai 🎉")

Selesai 🎉
